In [1]:
import pandas as pd

In [2]:
df = pd.read_json("profile_out/phone_1_run_1_events.json", lines=False)
df

,name,cpu_time_us,cuda_time_us,input_shapes,args,is_async,scope,fwd_thread_id
0,device_run,9411317.335,0,[],{},False,7,NaN
1,aten::full,2519.118,0,"[[], [], [], [], [], []]",{},False,0,NaN
2,aten::empty,5.917,0,"[[], [], [], [], [], []]",{},False,0,NaN
3,aten::fill_,2498.618,0,"[[1, 512], []]",{},False,0,NaN
4,aten::empty,2.916,0,"[[], [], [], [], [], []]",{},False,0,NaN
...,...,...,...,...,...,...,...,...
1002126,aten::zero_,0.292,0,[[512]],{},False,0,NaN
1002127,aten::zero_,0.375,0,[[512]],{},False,0,NaN
1002128,aten::zero_,0.292,0,[[512]],{},False,0,NaN
1002129,aten::zero_,0.333,0,[[512]],{},False,0,NaN


In [10]:
df_group = df.copy()
df_group["input_shapes"] = df_group["input_shapes"].apply(lambda x: str(x))
df_group = df_group.groupby(["name", "input_shapes"])["cpu_time_us"].sum().reset_index()

In [16]:
df_group[df_group["cpu_time_us"] > 0]

,name,input_shapes,cpu_time_us
0,_CopyToModelParallelRegion,"[[1, 1, 2048]]",55564.638
1,_CopyToModelParallelRegion,"[[1, 25, 2048]]",311.624
2,_GatherFromModelParallelRegion,"[[1, 1, 128256]]",1220.277
3,_GatherFromModelParallelRegion,"[[1, 25, 128256]]",7.083
4,_ReduceFromModelParallelRegion,"[[1, 1, 2048]]",20443.929
...,...,...,...
10231,aten::where,"[[1], [1], [1]]",10095.266
10232,aten::zero_,"[[25, 0]]",0.417
10233,aten::zero_,[[512]],23.415
10234,aten::zeros,"[[], [], [], [], []]",3.208


In [21]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
import numpy as np

# Prepare features
df_group['input_shapes_str'] = df_group['input_shapes'].apply(str)
X = df_group[['name', 'input_shapes_str']]
y = df_group['cpu_time_us']

# Preprocessing: One-hot encode 'name' and 'input_shapes_str'
preprocessor = ColumnTransformer([
    ('name', OneHotEncoder(handle_unknown='ignore'), ['name']),
    ('input_shapes', OneHotEncoder(handle_unknown='ignore'), ['input_shapes_str'])
])

# Model pipeline
model = make_pipeline(preprocessor, RandomForestRegressor(n_estimators=50, random_state=42))

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit model
model.fit(X_train, y_train)

# Example prediction
def predict_cpu_time(op_name, input_shape):
    input_shape_str = str(input_shape)
    X_pred = pd.DataFrame({'name': [op_name], 'input_shapes_str': [input_shape_str]})
    return model.predict(X_pred)[0]

# Example usage:
# predicted_time = predict_cpu_time('aten::zero_', [[512]])

In [26]:
predicted_time = predict_cpu_time('aten::zero_', [[512]])
predicted_time

np.float64(13.07495999750281)

In [36]:
df.groupby("name").size().sort_values(ascending=False)[:20]

name
aten::as_strided              122628
aten::view                     76001
aten::copy_                    62740
aten::slice                    60161
aten::to                       58884
aten::empty_strided            51715
aten::empty                    43286
aten::_to_copy                 41731
aten::reshape                  34304
aten::item                     31488
aten::_local_scalar_dense      31488
aten::expand                   30211
aten::mul                      29184
aten::linear                   28928
aten::view_as                  20992
aten::type_as                  20737
_CopyToModelParallelRegion     20736
aten::transpose                20480
aten::empty_like               17936
aten::add                      16656
dtype: int64